In [ ]:
import random, yaml, cv2
from pathlib import Path
import matplotlib.pyplot as plt

SRC = Path("/kaggle/input/datasets/xavierlermusieaux/kitti-yolo/yolo")
DST = Path("/kaggle/working/kitti_ped_cyc")
SEED, BLOCK = 42, 50
RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}
KEEP = {"pedestrian": 0, "cyclist": 1}

# 1. read class ids from data.yaml
names = yaml.safe_load(open(SRC / "data.yaml"))["names"]
if isinstance(names, list):
    names = dict(enumerate(names))
old2new = {int(i): KEEP[str(n).lower()] for i, n in names.items() if str(n).lower() in KEEP}
print("Original classes:", names)
print("Remap:", old2new)

# 2. pool train+val images, sorted by name
items = []
for sub in ("train", "val"):
    for img in (SRC / "images" / sub).iterdir():
        if img.suffix.lower() in {".png", ".jpg", ".jpeg"}:
            items.append((img, SRC / "labels" / sub / (img.stem + ".txt")))
items.sort(key=lambda t: t[0].stem)
print("Pooled images:", len(items))

# 3. split by blocks of 50 consecutive frames
blocks = [items[i:i + BLOCK] for i in range(0, len(items), BLOCK)]
random.Random(SEED).shuffle(blocks)
n_tr, n_va = round(len(blocks) * 0.70), round(len(blocks) * 0.15)
parts = {"train": blocks[:n_tr], "val": blocks[n_tr:n_tr + n_va], "test": blocks[n_tr + n_va:]}

# 4. write images (links) + remapped labels
for split, blks in parts.items():
    its = [it for b in blks for it in b]
    (DST / "images" / split).mkdir(parents=True, exist_ok=True)
    (DST / "labels" / split).mkdir(parents=True, exist_ok=True)
    counts, empty = [0, 0], 0
    for img, lbl in its:
        link = DST / "images" / split / img.name
        if not link.exists():
            link.symlink_to(img)
        out = []
        if lbl.exists():
            for line in lbl.read_text().splitlines():
                p = line.split()
                if len(p) == 5 and int(float(p[0])) in old2new:
                    new = old2new[int(float(p[0]))]
                    out.append(" ".join([str(new)] + p[1:]))
                    counts[new] += 1
        if not out:
            empty += 1
        (DST / "labels" / split / (img.stem + ".txt")).write_text("\n".join(out))
    print(f"{split}: {len(its)} images | pedestrians={counts[0]} cyclists={counts[1]} | without target={empty}")

# 5. data.yaml for YOLO
cfg = {"path": str(DST), "train": "images/train", "val": "images/val",
       "test": "images/test", "names": {0: "pedestrian", 1: "cyclist"}}
yaml.safe_dump(cfg, open(DST / "data.yaml", "w"), sort_keys=False)

# 6. annotation check: draw boxes on 6 random training images
imgs = sorted((DST / "images" / "train").iterdir())
random.Random(SEED).shuffle(imgs)
shown = 0
plt.figure(figsize=(16, 9))
for p in imgs:
    lines = (DST / "labels" / "train" / (p.stem + ".txt")).read_text().splitlines()
    if not lines:
        continue
    im = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    H, W = im.shape[:2]
    for ln in lines:
        c, xc, yc, w, h = ln.split()
        xc, yc, w, h = float(xc) * W, float(yc) * H, float(w) * W, float(h) * H
        col = (0, 255, 0) if c == "0" else (255, 0, 0)
        cv2.rectangle(im, (int(xc - w / 2), int(yc - h / 2)), (int(xc + w / 2), int(yc + h / 2)), col, 2)
    shown += 1
    plt.subplot(3, 2, shown); plt.imshow(im); plt.axis("off")
    if shown == 6:
        break
plt.show()

Original classes: {0: 'Car', 1: 'Truck', 2: 'Pedestrian', 3: 'Cyclist'}
Remap: {2: 0, 3: 1}
Pooled images: 7481
train: 5231 images | pedestrians=3178 cyclists=1163 | without target=3474
val: 1100 images | pedestrians=611 cyclists=225 | without target=765
test: 1150 images | pedestrians=698 cyclists=239 | without target=756
